In [1]:
# ============================================================
# Cell 1 — Imports and project paths
# ============================================================

from pathlib import Path

import ee
import geemap
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
BOUNDARY_PATH = DATA_DIR / "kenya_boundaries.gpkg"

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Boundary exists:", BOUNDARY_PATH.exists())
print("Raw data folder:", RAW_DIR)

Project root: c:\Projects\floodske\floodske
Boundary exists: True
Raw data folder: c:\Projects\floodske\floodske\data\raw


In [2]:
# ============================================================
# Cell 2 — Authenticate and initialize Earth Engine
# ============================================================

GEE_PROJECT = "august-analyze"

try:
    ee.Initialize(project=GEE_PROJECT)
    print("Earth Engine initialized.")
except Exception as error:
    print("Authentication required.")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("Earth Engine authenticated and initialized.")

print("GEE project:", GEE_PROJECT)

Earth Engine initialized.
GEE project: august-analyze


In [3]:
# ============================================================
# Cell 3 — Load Kenya boundary locally and prepare bounds
# ============================================================

kenya_gdf = gpd.read_file(BOUNDARY_PATH)

print("Records:", len(kenya_gdf))
print("CRS:", kenya_gdf.crs)
print("Columns:", kenya_gdf.columns.tolist())

kenya_gdf = kenya_gdf.to_crs("EPSG:4326")

minx, miny, maxx, maxy = kenya_gdf.total_bounds

print("Kenya bounds:")
print(minx, miny, maxx, maxy)

Records: 1
CRS: EPSG:4326
Columns: ['OBJECTID', 'iso2', 'iso3', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'cod_version', 'area_sqkm', 'lang', 'lang1', 'lang2', 'lang3', 'adm0_ref_name', 'center_lat', 'center_lon', 'geometry']
Kenya bounds:
33.91028865499999 -4.676879883000026 41.90602145899999 5.414123534999987


In [5]:
# ============================================================
# Cell 4 — Convert Kenya boundary to Earth Engine geometry
# ============================================================

kenya_ee = geemap.geopandas_to_ee(kenya_gdf)
kenya_geometry = kenya_ee.geometry()

print("Kenya boundary converted to Earth Engine geometry.")
print("Feature count:", kenya_ee.size().getInfo())

Kenya boundary converted to Earth Engine geometry.
Feature count: 1


In [6]:
# ============================================================
# Cell 5 — Define export settings
# ============================================================

EXPORT_FOLDER = "GEE_Kenya_Flood_Susceptibility"

EXPORT_SCALE = 90  # meters; good national-scale starting point
EXPORT_CRS = "EPSG:4326"

print("Export folder:", EXPORT_FOLDER)
print("Export scale:", EXPORT_SCALE)
print("Export CRS:", EXPORT_CRS)

Export folder: GEE_Kenya_Flood_Susceptibility
Export scale: 90
Export CRS: EPSG:4326


In [7]:
# ============================================================
# Cell 6 — Load DEM and create slope layer
# ============================================================

dem = (
    ee.Image("USGS/SRTMGL1_003")
    .select("elevation")
    .clip(kenya_geometry)
)

slope = (
    ee.Terrain.slope(dem)
    .rename("slope_degrees")
    .clip(kenya_geometry)
)

print("DEM and slope layers created.")

DEM and slope layers created.


In [8]:
# ============================================================
# Cell 7 — Load rainfall factor from CHIRPS
# ============================================================

rainfall = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate("2000-01-01", "2025-12-31")
    .filterBounds(kenya_geometry)
    .select("precipitation")
    .sum()
    .rename("rainfall_total_2000_2025_mm")
    .clip(kenya_geometry)
)

print("Rainfall factor created from CHIRPS daily precipitation.")

Rainfall factor created from CHIRPS daily precipitation.


In [9]:
# ============================================================
# Cell 8 — Load land cover factor
# ============================================================

landcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .rename("esa_worldcover_2021")
    .clip(kenya_geometry)
)

print("ESA WorldCover land cover factor created.")

ESA WorldCover land cover factor created.


In [10]:
# ============================================================
# Cell 9 — Create distance-to-water factor
# ============================================================

water = (
    landcover
    .eq(80)  # ESA WorldCover class 80 = permanent water
    .selfMask()
    .rename("water_mask")
)

distance_to_water = (
    water
    .fastDistanceTransform(1024)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename("distance_to_water_m")
    .clip(kenya_geometry)
)

print("Distance-to-water flood factor created.")

Distance-to-water flood factor created.


In [11]:
# ============================================================
# Cell 10 — Create export helper function
# ============================================================

def export_image_to_drive(image, description, file_name_prefix):
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=EXPORT_FOLDER,
        fileNamePrefix=file_name_prefix,
        region=kenya_geometry,
        scale=EXPORT_SCALE,
        crs=EXPORT_CRS,
        maxPixels=1e13,
    )
    task.start()
    print(f"Started export task: {description}")
    return task

In [ ]:
# ============================================================
# Cell 11 — Submit factor export tasks
# ============================================================

tasks = []

tasks.append(export_image_to_drive(
    dem,
    "kenya_dem_srtm_90m",
    "kenya_dem_srtm_90m"
))

tasks.append(export_image_to_drive(
    slope,
    "kenya_slope_srtm_90m",
    "kenya_slope_srtm_90m"
))

tasks.append(export_image_to_drive(
    rainfall,
    "kenya_chirps_rainfall_2000_2025_90m",
    "kenya_chirps_rainfall_2000_2025_90m"
))

tasks.append(export_image_to_drive(
    landcover,
    "kenya_esa_worldcover_2021_90m",
    "kenya_esa_worldcover_2021_90m"
))

tasks.append(export_image_to_drive(
    distance_to_water,
    "kenya_distance_to_water_90m",
    "kenya_distance_to_water_90m"
))

print("Submitted tasks:", len(tasks))